# Notebook 3 — Visualizing the Optimization Trajectory

Load the candidates collected in Notebook 2 and produce:

| Chart | What it shows |
|---|---|
| **Line chart** | Key metrics (CNS MPO, QED, BBB probability) across rounds |
| **Radar chart** | ADMET fingerprint of start vs best candidate |
| **Molecule grid** | Structural drawings of all candidates |
| **Summary table** | Final leaderboard with colour coding |

In [ ]:
import sys, os, json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    from rdkit.Chem.Draw import rdMolDraw2D
    from IPython.display import display, Image
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False
    print('RDKit not available — molecule grid will be skipped')

print('Imports OK')

---
## 1. Load Results from Notebook 2

In [ ]:
RESULTS_FILE = '../candidates.json'

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        candidates = json.load(f)
    print(f'Loaded {len(candidates)} candidates from {RESULTS_FILE}')
else:
    # Fallback: paste candidates list directly if file not found
    print('candidates.json not found — using example data.')
    print('Run notebook 02 first, or paste candidates list here:')
    candidates = []   # ← paste output from NB02 here if needed

if candidates:
    df = pd.DataFrame(candidates)
    print(f'\nColumns: {list(df.columns)}')
    print(df[['round', 'input_smiles', 'qed_score', 'cns_mpo_score', 'bbb_penetrates']].to_string())

---
## 2. Optimization Trajectory — Line Chart

How the key metrics evolved across each optimization round.

In [ ]:
if not candidates:
    print('No data — run notebook 02 first.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Lead Optimization Trajectory', fontsize=14, fontweight='bold', y=1.02)

    rounds = df['round'].tolist()
    colors = ['#2196F3', '#4CAF50', '#FF9800']

    metrics = [
        ('cns_mpo_score',   'CNS MPO Score',     '#a855f7', 4.0,  'CNS target (>4)'),
        ('qed_score',       'QED Score',          '#22c55e', 0.5,  'Min QED (>0.5)'),
        ('bbb_probability', 'BBB Probability',    '#38bdf8', 0.5,  'BBB threshold'),
    ]

    for ax, (col, title, color, target, target_label) in zip(axes, metrics):
        if col not in df.columns:
            ax.set_title(f'{title}\n(no data)')
            continue

        values = df[col].tolist()
        ax.plot(rounds, values, 'o-', color=color, linewidth=2, markersize=8, label=title)
        ax.axhline(y=target, color='red', linestyle='--', alpha=0.6, label=target_label)
        ax.fill_between(rounds, values, alpha=0.1, color=color)

        # Annotate each point
        for r, v in zip(rounds, values):
            ax.annotate(f'{v:.2f}', (r, v), textcoords='offset points',
                       xytext=(0, 8), ha='center', fontsize=8)

        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Optimization Round')
        ax.set_ylabel(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(rounds)

    plt.tight_layout()
    plt.savefig('../optimization_trajectory.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: optimization_trajectory.png')

---
## 3. ADMET Radar Chart — Start vs Best

Spider chart comparing the ADMET fingerprint of the starting molecule vs the best optimized candidate.

In [ ]:
if not candidates or len(candidates) < 2:
    print('Need at least 2 candidates for comparison.')
else:
    start = df.iloc[0]
    best  = df.loc[df['cns_mpo_score'].idxmax()] if 'cns_mpo_score' in df.columns else df.iloc[-1]

    # (column, display_name, min_val, max_val, higher_is_better)
    radar_props = [
        ('qed_score',       'QED',        0, 1,    True),
        ('bbb_probability', 'BBB Prob',   0, 1,    True),
        ('cns_mpo_score',   'CNS MPO',    0, 6,    True),
        ('gi_absorption',   'GI Absorb',  0, 1,    True),
        ('log_s',           'Solubility',-10, 2,   True),
        ('num_alerts',      'No Alerts',  0, 5,    False),
    ]

    def get_norm(row, col, min_v, max_v, higher_better):
        raw = row.get(col, 0)
        if col == 'gi_absorption':
            raw = {'High': 1.0, 'Moderate': 0.6, 'Low': 0.2}.get(str(raw), 0.5)
        try:
            raw = float(raw) if raw is not None else 0
        except (TypeError, ValueError):
            raw = 0
        n = (raw - min_v) / (max_v - min_v + 1e-9)
        n = max(0.0, min(1.0, n))
        return n if higher_better else 1.0 - n

    labels    = [p[1] for p in radar_props]
    start_vals = [get_norm(start, p[0], p[2], p[3], p[4]) for p in radar_props]
    best_vals  = [get_norm(best,  p[0], p[2], p[3], p[4]) for p in radar_props]

    # Close the polygon
    labels     += [labels[0]]
    start_vals += [start_vals[0]]
    best_vals  += [best_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=start_vals, theta=labels, fill='toself', name='Atenolol (start)',
        line_color='#ef4444', fillcolor='rgba(239,68,68,0.15)'
    ))
    fig.add_trace(go.Scatterpolar(
        r=best_vals, theta=labels, fill='toself',
        name=f'Best Candidate (Round {best["round"]})',
        line_color='#22c55e', fillcolor='rgba(34,197,94,0.15)'
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title='ADMET Profile: Atenolol vs Best Candidate',
        legend=dict(x=0.85, y=0.95),
        width=550, height=500,
    )
    fig.write_image('../admet_radar.png')
    fig.show()
    print('Saved: admet_radar.png')

---
## 4. Molecule Structure Grid

Visual comparison of all candidate molecules drawn with RDKit.

In [ ]:
if not RDKIT_OK:
    print('RDKit not available — skipping molecule grid.')
elif not candidates:
    print('No candidates loaded.')
else:
    mols, legends = [], []

    for c in candidates:
        smiles = c.get('canonical_smiles') or c.get('input_smiles', '')
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            mols.append(mol)
            label = (
                f"Round {c.get('round', '?')}\n"
                f"QED={c.get('qed_score', '?'):.2f}  "
                f"CNS={c.get('cns_mpo_score', '?'):.1f}\n"
                f"BBB={'✓' if c.get('bbb_penetrates') else '✗'}  "
                f"{c.get('decision', '')}"
            )
            legends.append(label)

    if mols:
        ncols = min(4, len(mols))
        img = Draw.MolsToGridImage(
            mols,
            molsPerRow=ncols,
            subImgSize=(300, 250),
            legends=legends,
        )
        img.save('../molecule_grid.png')
        display(img)
        print(f'Molecule grid saved: molecule_grid.png ({len(mols)} molecules)')
    else:
        print('No valid molecules to draw.')

---
## 5. Final Leaderboard

In [ ]:
if not candidates:
    print('No candidates loaded.')
else:
    leaderboard_cols = [
        'round', 'molecular_weight', 'clogp', 'qed_score',
        'bbb_penetrates', 'bbb_probability', 'cns_mpo_score',
        'cns_class', 'num_alerts', 'decision'
    ]
    cols = [c for c in leaderboard_cols if c in df.columns]
    lb = df[cols].copy()

    # Sort by CNS MPO descending
    if 'cns_mpo_score' in lb.columns:
        lb = lb.sort_values('cns_mpo_score', ascending=False)

    def color_row(row):
        decision = row.get('decision', '')
        color_map = {
            'Progress': 'background-color: #d4edda; color: #155724',
            'Optimize': 'background-color: #fff3cd; color: #856404',
            'Kill':     'background-color: #f8d7da; color: #721c24',
        }
        return [color_map.get(decision, '')] * len(row)

    styled = lb.style.apply(color_row, axis=1)
    display(styled)

    # Print the winner
    if 'cns_mpo_score' in df.columns:
        winner = df.loc[df['cns_mpo_score'].idxmax()]
        print('\n🏆 Best Candidate')
        print(f'   SMILES      : {winner.get("canonical_smiles", winner.get("input_smiles", "N/A"))}')
        print(f'   CNS MPO     : {winner.get("cns_mpo_score", "N/A")}')
        print(f'   QED         : {winner.get("qed_score", "N/A")}')
        print(f'   BBB+        : {winner.get("bbb_penetrates", "N/A")}')
        print(f'   Alerts      : {winner.get("num_alerts", "N/A")}')
        print(f'   Decision    : {winner.get("decision", "N/A")}')

---
## Summary

### What this project demonstrates

| Skill | How it shows |
|---|---|
| **Tool use / agentic AI** | Agent autonomously calls 3 tools in a loop |
| **Domain expertise** | System prompt encodes real medicinal chemistry knowledge |
| **API integration** | Existing production ADMET API reused as a tool |
| **Iterative reasoning** | Agent backtracks and improves — not a one-shot prompt |
| **Transparency** | Every step printed; full conversation history saved |
| **Visualization** | Results communicated with charts, not just numbers |

### Architecture recap

```
User goal
   │
   ▼
Claude (claude-opus-4-6)
  ├── validate_smiles()      ← RDKit (local)
  ├── analyze_molecule()     ← Drug Discovery Triage API (production)
  └── compare_candidates()   ← Drug Discovery Triage API × N
          │
          └─► Best candidate + chemical reasoning
```